In [ ]:
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [11]:
MODEL = "llama3.1:8b"
DB_NAME = "vector_db"

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [25]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model=MODEL, temperature=0)

# retriever.invoke("Who is Lisa?")
# llm.invoke("Who is Lisa?")

In [26]:
SYSTEM_PROMPT_TEMPLATE = """
  You are a knowledgeable, friendly assistant representing the company Insurellm.
  You are chatting with a user about Insurellm.
  If relevant, use the given context to answer any question.
  If you don't know the answer, say so.

  Context:
  {context}
"""

def answer_question(question: str, history):
  docs = retriever.invoke(question)
  context = "\n\n".join(doc.page_content for doc in docs)
  system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
  response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
  return response.content

# answer_question("Who is Lisa Anderson?", [])

In [27]:
gr.ChatInterface(answer_question).launch(inbrowser=True)

c:\Users\maaro\Documents\My Projects\llm-engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
